In [8]:
import json
import os
import pandas as pd
from typing import List
from pydantic import BaseModel, Field
from openai import OpenAI
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split

In [9]:
# Load environment variables from .env file
load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")

# Initialize OpenAI client configured for OpenRouter API
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    timeout=30.0
)
os.makedirs("data", exist_ok=True)

In [10]:
# Define Data Models for Structured Output Validation
class NERSample(BaseModel):
    sentence: str = Field(description="The full generated sentence containing mountain names.")
    tokens: List[str] = Field(description="Tokens of the sentence split by spaces.")
    ner_tags: List[str] = Field(description="BIO tags corresponding to each token (O, B-MOUNTAIN, I-MOUNTAIN).")

class NERDataset(BaseModel):
    samples: List[NERSample]

# System instructions enforcing BIO schema, tokenization rules, and JSON output
SYSTEM_PROMPT = f"""
You are an expert NLP Data Scientist creating a high-quality Named Entity Recognition (NER) dataset.
Your task is to generate natural English sentences that contain mountain names (e.g., Mount Everest, K2, Matterhorn, Hoverla, Ben Nevis).

Guidelines:
1. Generate diverse contexts: travel blogs, geography books, mountaineering news, conversations, historical facts.
2. Include both single-mountain and multi-mountain sentences (e.g., comparing two or three peaks).
3. Split the sentence into tokens by whitespace.
4. Provide correct BIO format NER tags for every token:
   - "O" for non-mountain tokens.
   - "B-MOUNTAIN" for the first token of a mountain entity.
   - "I-MOUNTAIN" for subsequent tokens of a multi-word mountain entity.
5. Ensure exact length match: len(tokens) MUST equal len(ner_tags).
6. Strip or isolate punctuation: Ensure punctuation marks (like '.', ',', '!') are separated by spaces as individual tokens or removed from sentence generation entirely. Example good tokens: ["Traveling", "to", "Mount", "Everest", "."]

CRITICAL REQUIREMENT:
You MUST respond strictly with a valid JSON object matching this schema:
{json.dumps(NERDataset.model_json_schema(), indent=2)}
"""

In [11]:
def generate_batch_via_openrouter(batch_size: int = 10) -> List[dict]:
    """Requests a single batch of NER samples from the LLM and validates token-tag alignment"""
    completion = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Generate {batch_size} diverse sentences with mountain entities and BIO tagging."}
        ],
        response_format={"type": "json_object"},
        temperature=0.8,
        extra_headers={
            "HTTP-Referer": "https://github.com",
            "X-Title": "NER Mountain Dataset Generator"
        }
    )

    # Parse JSON response using Pydantic schema
    content = completion.choices[0].message.content
    data = json.loads(content)
    parsed = NERDataset(**data)

    # Filter out samples where tokens and tags lengths do not match
    batch_data = []
    for item in parsed.samples:
        if len(item.tokens) == len(item.ner_tags):
            batch_data.append({
                "sentence": item.sentence,
                "tokens": item.tokens,
                "ner_tags": item.ner_tags
            })

    return batch_data

In [12]:
def generate_full_openrouter_dataset(total_samples: int = 100, batch_size: int = 20) -> List[dict]:
    """Generates the full dataset in batches with automatic retry on failure."""
    dataset = []
    print(f"Starting dataset generation via OpenRouter API (Target: {total_samples} samples)...")

    while len(dataset) < total_samples:
        needed = min(batch_size, total_samples - len(dataset))
        try:
            batch = generate_batch_via_openrouter(batch_size=needed)
            dataset.extend(batch)
            print(f"  Progress: {len(dataset)}/{total_samples} samples generated.")
        except Exception as e:
            print(f"  Error during generation: {e}. Retrying...")

    return dataset

In [13]:
# Generate total required samples
TOTAL_SAMPLES = 1000
full_data = generate_full_openrouter_dataset(total_samples=TOTAL_SAMPLES, batch_size=20)

# Save raw full dataset
with open("data/dataset.json", "w", encoding="utf-8") as f:
    json.dump(full_data, f, ensure_ascii=False, indent=2)

# Perform train-validation split
train_data, val_data = train_test_split(full_data, test_size=0.2, random_state=42)

# Save split datasets
with open("data/train.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

with open("data/val.json", "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)

print(f"\nGeneration Complete!")
print(f"Total samples: {len(full_data)} | Train: {len(train_data)} | Val: {len(val_data)}")

Starting dataset generation via OpenRouter API (Target: 1000 samples)...
  Progress: 13/1000 samples generated.
  Progress: 27/1000 samples generated.
  Progress: 44/1000 samples generated.
  Progress: 62/1000 samples generated.
  Progress: 79/1000 samples generated.
  Progress: 90/1000 samples generated.
  Progress: 106/1000 samples generated.
  Progress: 122/1000 samples generated.
  Progress: 140/1000 samples generated.
  Progress: 158/1000 samples generated.
  Progress: 172/1000 samples generated.
  Progress: 187/1000 samples generated.
  Progress: 200/1000 samples generated.
  Progress: 212/1000 samples generated.
  Progress: 229/1000 samples generated.
  Progress: 243/1000 samples generated.
  Progress: 259/1000 samples generated.
  Progress: 276/1000 samples generated.
  Progress: 289/1000 samples generated.
  Progress: 304/1000 samples generated.
  Progress: 318/1000 samples generated.
  Progress: 324/1000 samples generated.
  Progress: 341/1000 samples generated.
  Progress: 3

In [14]:
# Display a formatted preview of generated samples
print("=" * 80)
print(" DATASET PREVIEW")
print("=" * 80)

for idx, sample in enumerate(full_data[:3], 1):
    print(f"\nSample #{idx}:")
    print(f"Full Sentence: \"{sample['sentence']}\"")

    df = pd.DataFrame({
        "Token": sample["tokens"],
        "BIO Tag": sample["ner_tags"]
    })

    entities = [f"{t} ({tag})" for t, tag in zip(sample["tokens"], sample["ner_tags"]) if tag != "O"]
    print(f"Entities Found: {', '.join(entities) if entities else 'None'}")
    print("\nToken Level Breakdown:")
    print(df.to_string(index=False))
    print("-" * 60)

 DATASET PREVIEW

Sample #1:
Full Sentence: "Hiking to Mount Everest is a dream for many adventurers."
Entities Found: Mount (B-MOUNTAIN), Everest (I-MOUNTAIN)

Token Level Breakdown:
      Token    BIO Tag
     Hiking          O
         to          O
      Mount B-MOUNTAIN
    Everest I-MOUNTAIN
         is          O
          a          O
      dream          O
        for          O
       many          O
adventurers          O
          .          O
------------------------------------------------------------

Sample #2:
Full Sentence: "The majestic Matterhorn stands tall in the Swiss Alps."
Entities Found: Matterhorn (B-MOUNTAIN)

Token Level Breakdown:
     Token    BIO Tag
       The          O
  majestic          O
Matterhorn B-MOUNTAIN
    stands          O
      tall          O
        in          O
       the          O
     Swiss          O
      Alps          O
         .          O
------------------------------------------------------------

Sample #3:
Full Sentence: "